In [0]:
%sql
CREATE OR REPLACE TABLE silver.crm_cust_info (
  cst_id INT,
  cst_key VARCHAR(50),
  cst_firstname VARCHAR(50),
  cst_lastname VARCHAR(50),
  cst_marital_status VARCHAR(50),
  cst_gndr VARCHAR(50),
  cst_create_date DATE,
  dwh_create_date TIMESTAMP 
);

In [0]:
%sql
TRUNCATE TABLE silver.crm_cust_info;
INSERT INTO silver.crm_cust_info 
(cst_id,
cst_key,
cst_firstname,
cst_lastname,
cst_gndr,
cst_marital_status,
cst_create_date,
dwh_create_date)

SELECT 
cst_id, 
cst_key,
TRIM(cst_firstname) AS cst_firstname,    -- Remove unwated space
TRIM(cst_lastname) AS cst_lastname, 
CASE WHEN UPPER(TRIM(cst_gndr)) = 'F' THEN 'Female'        -- Data Standardization & Consistency 
     WHEN UPPER(TRIM(cst_gndr)) = 'M' Then 'Male'
     ELSE 'n/a'
END cst_gndr,
CASE  WHEN UPPER(TRIM(cst_marital_status)) = 'M' THEN 'Married'
      WHEN UPPER(TRIM(cst_marital_status)) = 'S' THEN 'Single'
      ELSE 'n/a'
END cst_marital_status, 
DATE(cst_create_date),                                     
current_timestamp() AS dwh_create_date
FROM 
(
  SELECT * FROM 
(
SELECT *, ROW_NUMBER() OVER (PARTITION BY cst_id ORDER BY cst_create_date DESC) AS flag_last  
FROM bronze.crm_cust_info                    -- Data Quality: Remove duplicate & null value in PK 
WHERE cst_id IS NOT NULL
)
WHERE flag_last = 1 
);

SELECT * FROM  silver.crm_cust_info

In [0]:
%sql
CREATE OR REPLACE TABLE silver.crm_prd_info (
  prd_id INT,
  cat_id VARCHAR(50),
  prd_key VARCHAR(50),
  prd_nm VARCHAR(50),
  prd_cost DECIMAL (10,2), 
  prd_line VARCHAR (50),
  prd_start_dt DATE,
  prd_end_dt DATE,
  dwh_create_date TIMESTAMP
); 

In [0]:
%sql
TRUNCATE TABLE silver.crm_prd_info;
INSERT INTO silver.crm_prd_info (prd_id, cat_id, prd_key, prd_nm, prd_cost, prd_line, prd_start_dt, prd_end_dt, dwh_create_date)
SELECT
prd_id,
REPLACE(SUBSTRING(prd_key, 1, 5), '-','_') AS cat_id,  -- Data Transformation to match with cat_id in px_cat_g1v2 table
SUBSTRING(prd_key, 7, length(prd_key)) AS prd_key,
prd_nm,
COALESCE(prd_cost, 0) AS prd_cost,
CASE WHEN TRIM(UPPER(prd_line))= 'M' THEN 'Mountain'
     WHEN TRIM(UPPER(prd_line))= 'R' THEN 'Road'
     WHEN TRIM(UPPER(prd_line))= 'T' THEN 'Touring'
     WHEN TRIM(UPPER(prd_line))= 'S' THEN 'other Sales'
     ELSE 'n/a'
END prd_line,
CAST(prd_start_dt AS DATE),
LEAD(CAST(prd_start_dt AS DATE)) OVER (PARTITION BY prd_key ORDER BY prd_start_dt) -1 AS prd_end_dt,
CURRENT_TIMESTAMP AS dwh_create_date
FROM bronze.crm_prd_info; 

SELECT * FROM silver.crm_prd_info


In [0]:
%sql
CREATE OR REPLACE TABLE silver.crm_sales_details ( 
  sls_ord_num VARCHAR(50), 
  sls_prd_key VARCHAR(50), 
  sls_cust_id INT,
  sls_order_dt DATE,
  sls_ship_dt DATE,
  sls_due_dt DATE,
  sls_sales DECIMAL (10,2),
  sls_quantity INT,
  sls_price DECIMAL (10,2),
  dwh_create_date TIMESTAMP

); 
SELECT * FROM bronze.crm_sales_details

In [0]:
%sql
TRUNCATE TABLE silver.crm_sales_details;
INSERT INTO silver.crm_sales_details ( 
  sls_ord_num, 
  sls_prd_key, 
  sls_cust_id,
  sls_order_dt,
  sls_due_dt,
  sls_ship_dt,
  sls_quantity,
  sls_sales,
  sls_price, 
  dwh_create_date

)

SELECT sls_ord_num, 
       sls_prd_key, 
       sls_cust_id, 
       CASE WHEN sls_order_dt = 0 OR len(sls_order_dt) != 8 THEN NULL 
            ELSE TO_DATE(CAST(sls_order_dt AS VARCHAR(50)), 'yyyyMMdd')
       END sls_order_dt,
       CASE WHEN sls_due_dt = 0 OR len(sls_due_dt) != 8 THEN NULL  
            ELSE TO_DATE(CAST(sls_due_dt AS VARCHAR(50)), 'yyyyMMdd')
       END sls_due_dt,
       CASE WHEN sls_ship_dt = 0 OR len(sls_ship_dt) !=8 THEN NULL 
            ELSE TO_DATE(CAST(sls_ship_dt AS VARCHAR(50)), 'yyyyMMdd')
       END sls_ship_dt,
       sls_quantity,
       CASE WHEN sls_sales IS NULL OR sls_sales <= 0 OR sls_sales != sls_quantity * ABS(sls_price) 
        THEN sls_quantity * sls_price
     ELSE sls_sales 
     END sls_sales, 

     CASE WHEN sls_price = 0 OR sls_price IS NULL 
        THEN sls_sales / sls_quantity
     WHEN sls_price <0
        then abs(sls_price)
     ELSE sls_price
     end sls_price,
     current_timestamp() AS dwh_create_date 
FROM   bronze.crm_sales_details; 

SELECT * FROM silver.crm_sales_details


In [0]:
%sql
CREATE OR REPLACE TABLE silver.erp_cust_az12 (
  CID VARCHAR (50), 
  BDATE DATE,
  GEN VARCHAR(50),
  dwh_create_date TIMESTAMP

); 


In [0]:
%sql
TRUNCATE TABLE silver.erp_cust_az12;
INSERT INTO silver.erp_cust_az12 (CID, BDATE, GEN, dwh_create_date)
SELECT
CASE WHEN CID LIKE 'NAS%' THEN SUBSTRING(CID, 4, len(CID))
     ELSE CID
END CID,
CASE WHEN CAST (BDATE AS DATE) > CURRENT_DATE() THEN NULL  
     ELSE CAST (BDATE AS DATE)
END BDATE,
CASE WHEN UPPER(TRIM(GEN)) IN ('F', 'FEMALE') THEN 'Female'
     WHEN UPPER(TRIM(GEN)) IN ('M', 'MALE') THEN 'Male'
     ELSE 'n/a'
END GEN,
CURRENT_TIMESTAMP AS dwh_create_date
FROM bronze.erp_cust_az12;

SELECT * FROM silver.erp_cust_az12
 

In [0]:
%sql
 CREATE OR REPLACE TABLE silver.erp_loc_a101 (
  CID VARCHAR(50), 
  CNTRY VARCHAR(50),
  dwh_create_date TIMESTAMP
);
SELECT * FROM bronze.erp_loc_a101

In [0]:
%sql
TRUNCATE TABLE silver.erp_loc_a101;
INSERT INTO silver.erp_loc_a101 (CID, CNTRY, dwh_create_date) 
SELECT 
REPLACE(cid,'-',''),  -- remove dash to match with cust_info table 
CASE WHEN TRIM(cntry) IN ('USA', 'US') THEN 'United State'   -- standardization 
     WHEN TRIM(cntry) = '' OR cntry is NULL THEN 'n/a'
     WHEN TRIM(cntry) = 'DE' THEN 'Germany'
     ELSE cntry 
END cntry,
current_timestamp() AS dwh_create_date 
FROM bronze.erp_loc_a101;

SELECT * FROM silver.erp_loc_a101

In [0]:
%sql

CREATE OR REPLACE TABLE silver.erp_px_cat_g1v2 (
    ID VARCHAR(50),
    CAT VARCHAR(50),
    SUBCAT VARCHAR(50), 
    MAINTENANCE BOOLEAN,
    dwh_create_date TIMESTAMP
); 

In [0]:
%sql
TRUNCATE TABLE silver.erp_px_cat_g1v2;
INSERT INTO silver.erp_px_cat_g1v2 (id, cat, subcat, maintenance, dwh_create_date) 
(SELECT UPPER(TRIM(id)) AS id,
       TRIM(cat) AS CAT, 
       TRIM(subcat) AS SUBCAT, 
       TRIM(maintenance) AS MAINTENANCE,
       current_timestamp() AS dwh_create_date
FROM bronze.erp_px_cat_g1v2);
SELECT * FROM silver.erp_px_cat_g1v2